# G11project · Colab Demo-Lite Training

This notebook runs an independent highway-only course-demo experiment. It does not read, overwrite, or delete `G11project-formal`. Progress is stored in `G11project-demo-lite` and can be resumed after a Colab runtime restart. The output is preliminary course-demo evidence, not a paper-scale formal result.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/yangyu-rgb/G11project.git"
GIT_REF = "main"
REPOSITORY = Path("/content/G11project")
DEMO_LITE_ROOT = Path("/content/drive/MyDrive/G11project-demo-lite")
DEMO_LITE_WORK = Path("/content/g11-demo-lite-work")

if not (REPOSITORY / ".git").is_dir():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY)], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "checkout", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "pull", "--ff-only", "origin", GIT_REF], check=True)
DEMO_LITE_ROOT.mkdir(parents=True, exist_ok=True)
DEMO_LITE_WORK.mkdir(parents=True, exist_ok=True)
os.chdir(REPOSITORY)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Demo-lite Drive root:", DEMO_LITE_ROOT)

In [ ]:
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "sumo", "sumo-tools"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", "./BackEnd[dev]"], check=True)
os.environ["SUMO_HOME"] = "/usr/share/sumo"
sumo_tools = Path(os.environ["SUMO_HOME"]) / "tools"
os.environ["PYTHONPATH"] = str(sumo_tools) + os.pathsep + os.environ.get("PYTHONPATH", "")
print("SUMO tools:", sumo_tools)

In [ ]:
import shutil
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before starting demo-lite training"
assert shutil.which("sumo"), "SUMO installation failed"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
sumo_version = subprocess.run(
    ["sumo", "--version"],
    capture_output=True,
    text=True,
    check=True,
)
print(sumo_version.stdout.splitlines()[0])
subprocess.run(
    ["python", "-c", "import traci, sumolib; print('TraCI and sumolib imports verified')"],
    check=True,
)
subprocess.run(
    [
        "python",
        "BackEnd/scripts/run_demo_lite_pipeline.py",
        "--persistent-root",
        str(DEMO_LITE_ROOT),
        "--work-root",
        str(DEMO_LITE_WORK),
        "--status",
    ],
    check=True,
)

## Run or resume the three-stage pipeline

Run the next cell. It advances `training → comparison → results`. Return code 75 means a safe time-budget pause. After a runtime restart, rerun the setup cells and then this cell. Never run two runtimes against the same `DEMO_LITE_ROOT`.

In [ ]:
import json

while True:
    status_process = subprocess.run(
        [
            "python",
            "BackEnd/scripts/run_demo_lite_pipeline.py",
            "--persistent-root",
            str(DEMO_LITE_ROOT),
            "--work-root",
            str(DEMO_LITE_WORK),
            "--status",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    status = json.loads(status_process.stdout)
    print(json.dumps(status, indent=2, ensure_ascii=False))
    if status["next_stage"] is None:
        print("Demo-lite pipeline completed.")
        break
    result = subprocess.run(
        [
            "python",
            "BackEnd/scripts/run_demo_lite_pipeline.py",
            "--stage",
            "next",
            "--persistent-root",
            str(DEMO_LITE_ROOT),
            "--work-root",
            str(DEMO_LITE_WORK),
            "--time-budget-minutes",
            "360",
        ]
    )
    if result.returncode == 75:
        print("Safely paused. Rerun this cell to resume.")
        break
    if result.returncode != 0:
        raise RuntimeError(f"Demo-lite stage failed with return code {result.returncode}")

In [ ]:
subprocess.run(
    [
        "python",
        "BackEnd/scripts/run_demo_lite_pipeline.py",
        "--persistent-root",
        str(DEMO_LITE_ROOT),
        "--work-root",
        str(DEMO_LITE_WORK),
        "--status",
    ],
    check=True,
)

RESULTS = DEMO_LITE_ROOT / "presentation_results"
print("Presentation outputs:")
for path in sorted(RESULTS.glob("*")):
    print(" -", path)